In [2]:
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns

import Model_func as mf

# Collect Data

In [3]:
data_prod_path = '../../data/prod/eCO2mix_RTE_Auvergne-Rhone-Alpes_cleaned.csv'
data_prod = mf.data_coll_prod(data_prod_path)

data_sat_path = '../../data/LandSat/result_EarthExplorer_region_ARA.csv'
data_sat = mf.data_coll_landsat(data_sat_path)

data_weather_path = '../../data/openweathermap/merge_openweathermap_cleaned.csv'
data_weather = mf.data_coll_weather(data_weather_path)


In [4]:
data_weather.columns

Index(['dt', 'sunrise', 'sunset', 'temp', 'feels_like', 'pressure', 'humidity',
       'dew_point', 'clouds', 'wind_speed', 'wind_deg', 'rain', 'snow', 'city',
       'lat', 'lon', 'weather_main', 'weather_desc', 'Time'],
      dtype='object')

In [5]:
fig = px.scatter(
    data_sat,
    x='Time',
    y='Scene Cloud Cover L1',
    color='WRS Path',  # ça fonctionne ici
    title="Scene Cloud Cover L1 vs Time"
)


fig.show()


In [6]:
targeted_weather_data = mf.add_target_column_weather(data_weather, data_prod)
targeted_sat_data = mf.add_target_column_sat(data_sat, data_prod)


In [7]:
targeted_weather_data.head()

,Time,temp,pressure,humidity,clouds,tch_solaire_(%)
0,2021-01-09 11:00:00,0.778,1017.2,71.0,76.6,13.28
1,2021-01-10 11:00:00,0.264,1018.6,70.4,67.6,15.04
2,2021-01-11 11:00:00,-0.756,1024.2,72.2,36.6,16.81
3,2021-01-12 11:00:00,2.110,1025.0,78.4,92.6,9.24
4,2021-01-13 11:00:00,7.260,1024.6,91.6,84.4,7.82


In [9]:
targeted_weather_data.describe()

,Time,temp,pressure,humidity,clouds,tch_solaire_(%)
count,1770,1770.000000,1770.000000,1770.000000,1770.000000,1770.000000
mean,2023-06-12 22:23:41.694915328,15.362667,1018.104859,66.569266,49.980678,31.967605
min,2021-01-09 11:00:00,-0.756000,985.600000,29.600000,0.000000,2.310000
25%,2022-03-27 16:00:00,9.746500,1014.400000,57.200000,26.000000,20.282500
50%,2023-06-12 22:00:00,15.007000,1018.200000,67.600000,52.800000,32.175000
75%,2024-08-28 04:00:00,20.976500,1022.150000,76.350000,73.750000,43.610000
max,2025-11-13 11:00:00,33.998000,1041.200000,96.200000,100.000000,85.710000
std,NaN,7.318592,7.325669,12.896182,28.688925,14.848196


In [10]:
targeted_weather_data.head(30)

,Time,temp,pressure,humidity,clouds,tch_solaire_(%)
0,2021-01-09 11:00:00,0.778,1017.2,71.0,76.6,13.28
1,2021-01-10 11:00:00,0.264,1018.6,70.4,67.6,15.04
2,2021-01-11 11:00:00,-0.756,1024.2,72.2,36.6,16.81
3,2021-01-12 11:00:00,2.110,1025.0,78.4,92.6,9.24
4,2021-01-13 11:00:00,7.260,1024.6,91.6,84.4,7.82
5,2021-01-14 11:00:00,7.324,1022.6,81.2,75.4,17.65
6,2021-01-15 11:00:00,3.704,1023.0,85.6,85.8,5.38
7,2021-01-16 11:00:00,0.318,1028.4,72.2,60.6,21.43
8,2021-01-17 11:00:00,4.498,1024.0,86.2,75.0,11.85
9,2021-01-18 11:00:00,3.676,1029.8,84.6,50.2,21.51


In [26]:
data_weather['city'].unique()

array(['Moulins', 'Aurillac', 'Saint-Étienne', 'Annecy', 'Nyons'],
      dtype=object)

In [37]:
city = 'Moulins'

# data_weather[data_weather['city']==city]

def split_data_weather_by_cities(data_weather, Cities='city'):
    """
    Split the dataframe data_weather into 5 dataframes : 1 for each city.

    Returns a dictionary {"city" : dataframe}.
    """
    dict = {}
    for city in data_weather[Cities].unique():
        key_name = f"{city}"  # name of the dataframe=city
        dict[key_name] = data_weather[data_weather[Cities]==city].copy()
    return dict

tables = split_data_weather_by_cities(data_weather)

tables.keys()



dict_keys(['Moulins', 'Aurillac', 'Saint-Étienne', 'Annecy', 'Nyons'])

In [40]:
tables['Annecy'].shape

(1770, 19)

In [41]:

# Sort each DataFrame by 'Date' if the column exists
sorted_dict = {}
for name, df in dict.items():
    if 'Time' in df.columns:
        sorted_df = df.sort_values('Time').reset_index(drop=True)
        sorted_dict[name] = sorted_df
    else:
        raise ValueError(f"The DataFrame '{name}' does not contain a 'Time' column.")

# Concatenate horizontally with a prefix for each DataFrame
final_df = pd.concat(
    [df.add_prefix(f"{name}_") for name, df in sorted_dict.items()],
    axis=1
)

print(final_df.shape)  # Expected: (1770, 19*5 = 95)



TypeError: unbound method dict.items() needs an argument